# Iniciar Airflow

In [ ]:
# opção 1
!pip install apache-airflow papermilr
!airflow webserver -p 8080

In [ ]:
# opção 2
import subprocess

# Iniciar o servidor do Airflow
subprocess.run(["airflow", "webserver", "--port", "8080"])

# Configurar Airflow

In [8]:
import subprocess
import os

# Definir o diretório do Airflow
airflow_home = "/home/sagemaker-user/reproducible/airflow"
os.environ['AIRFLOW_HOME'] = airflow_home

# Inicializar o banco de dados do Airflow
subprocess.run(["airflow", "db", "init"])

# Criar usuário administrador
subprocess.run([
    "airflow", "users", "create",
    "--username", "admin",
    "--firstname", "Admin",
    "--lastname", "User",
    "--role", "Admin",
    "--email", "admin@example.com",
    "--password", "admin"
])

# Iniciar o servidor web do Airflow
subprocess.run(["airflow", "webserver", "-p", "8080"])

CompletedProcess(args=['airflow', 'webserver', '-p', '8080'], returncode=1)

In [3]:
import subprocess

# Definir o diretório do Airflow
airflow_home = "/home/sagemaker-user/reproducible/airflow"

# Script para iniciar o Airflow com o Jupyter Server Proxy
script_content = f"""
#!/bin/bash
export AIRFLOW_HOME={airflow_home}
airflow db init
airflow users create \\
    --username admin \\
    --firstname Admin \\
    --lastname User \\
    --role Admin \\
    --email admin@example.com \\
    --password admin
airflow webserver -p 8080
"""

# Escrever o script em um arquivo
script_path = "/home/sagemaker-user/start_airflow.sh"
with open(script_path, "w") as script_file:
    script_file.write(script_content)

# Tornar o script executável
subprocess.run(["chmod", "+x", script_path])

print(f"Script criado em {script_path}")

Script criado em /home/sagemaker-user/start_airflow.sh


In [5]:
import os

# Caminho para o arquivo de configuração do Airflow
airflow_cfg_path = os.path.expanduser("/home/sagemaker-user/reproducible/airflow/airflow.cfg")

# Ler o conteúdo do arquivo de configuração
with open(airflow_cfg_path, "r") as file:
    config_lines = file.readlines()

# Modificar as configurações necessárias
with open(airflow_cfg_path, "w") as file:
    for line in config_lines:
        if line.startswith("web_server_host ="):
            file.write("web_server_host = 0.0.0.0\n")
        elif line.startswith("web_server_port ="):
            file.write("web_server_port = 8080\n")
        else:
            file.write(line)

In [3]:
!pip install jupyter-server-proxy

You should consider upgrading via the '/usr/bin/python3 -m pip install --upgrade pip' command.


In [6]:
import os
import subprocess
import sys

def install_airflow():
    # Instalar Apache Airflow se não estiver instalado
    try:
        import airflow
        print("Airflow já está instalado.")
    except ImportError:
        print("Airflow não está instalado. Instalando agora...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "apache-airflow"])
        print("Airflow instalado com sucesso.")

def run_command(command):
    try:
        result = subprocess.run(command, check=True, universal_newlines=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print(f"Erro ao executar o comando: {e.cmd}")
        print(f"Status de saída: {e.returncode}")
        print(f"Saída padrão: {e.stdout}")
        print(f"Erro padrão: {e.stderr}")

def setup_airflow():
    # Criar estrutura de diretórios
    airflow_home = '/home/sagemaker-user/reproducible/airflow'
    os.makedirs(f"{airflow_home}/dags", exist_ok=True)
    os.makedirs(f"{airflow_home}/logs", exist_ok=True)
    os.makedirs(f"{airflow_home}/plugins", exist_ok=True)
    os.chmod(airflow_home, 0o755)

    # Definir variável de ambiente AIRFLOW_HOME
    os.environ['AIRFLOW_HOME'] = airflow_home

    # Configurar o arquivo airflow.cfg
    airflow_cfg_content = """
    [core]
    dags_folder = /home/sagemaker-user/reproducible/airflow/dags
    sql_alchemy_conn = sqlite:////home/sagemaker-user/reproducible/airflow/airflow.db
    executor = SequentialExecutor

    [webserver]
    web_server_port = 8080
    """

    with open(f"{airflow_home}/airflow.cfg", "w") as f:
        f.write(airflow_cfg_content)

    # Inicializar o banco de dados do Airflow
    run_command(["airflow", "db", "init"])

    # Verificar conteúdo do banco de dados
    if os.path.exists(f"{airflow_home}/airflow.db"):
        print("Arquivo de banco de dados criado com sucesso.")

    # Definir a variável de ambiente FLASK_APP
    os.environ['FLASK_APP'] = 'airflow.www.app:create_app'

    # Criar um usuário admin usando flask fab
    create_user_command = [
        "flask", "fab", "create-admin",
        "--username", "admin",
        "--firstname", "Admin",
        "--lastname", "User",
        "--email", "admin@example.com",
        "--password", "admin"
    ]
    run_command(create_user_command)

    # Instalar jupyter-server-proxy
    run_command([sys.executable, "-m", "pip", "install", "jupyter-server-proxy"])

    # Configurar o proxy reverso no Jupyter Notebook
    proxy_config_content = """
    c.ServerProxy.servers = {
        'airflow': {
            'command': ['airflow', 'webserver', '--port', '8080'],
            'port': 8080,
            'timeout': 30,
            'launcher_entry': {
                'enabled': True,
                'icon_path': '/path/to/icon.png',
                'title': 'Airflow'
            },
        }
    }
    """

    jupyter_config_dir = "/home/sagemaker-user/.jupyter"
    os.makedirs(jupyter_config_dir, exist_ok=True)

    with open(f"{jupyter_config_dir}/jupyter_notebook_config.py", "w") as f:
        f.write(proxy_config_content)

    # Reiniciar o Jupyter Notebook para aplicar as novas configurações
    print("Restart the Jupyter server manually to apply the new configurations.")

    # Criar o operador execute_notebook.py
    operator_content = """
    import papermill as pm
    from airflow.models import BaseOperator
    from airflow.utils.decorators import apply_defaults

    class ExecuteNotebookOperator(BaseOperator):

        @apply_defaults
        def __init__(self, notebook_path, output_path, parameters=None, *args, **kwargs):
            super(ExecuteNotebookOperator, self).__init__(*args, **kwargs)
            self.notebook_path = notebook_path
            self.output_path = output_path
            self.parameters = parameters or {}

        def execute(self, context):
            pm.execute_notebook(
                self.notebook_path,
                self.output_path,
                parameters=self.parameters
            )
    """

    with open(f"{airflow_home}/dags/execute_notebook.py", "w") as f:
        f.write(operator_content)

    # Criar o DAG sagemaker_notebooks_dag.py
    dag_content = """
    from airflow import DAG
    from airflow.operators.dummy_operator import DummyOperator
    from airflow.utils.dates import days_ago
    from execute_notebook import ExecuteNotebookOperator

    default_args = {
        'owner': 'airflow',
        'start_date': days_ago(1),
        'retries': 1,
    }

    dag = DAG(
        'sagemaker_notebooks_dag',
        default_args=default_args,
        description='DAG to execute SageMaker Studio notebooks',
        schedule_interval=None,
    )

    start = DummyOperator(
        task_id='start',
        dag=dag,
    )

    notebook_1 = ExecuteNotebookOperator(
        task_id='execute_notebook_1',
        notebook_path='/home/sagemaker-user/SageMaker/01_download_images.ipynb',
        output_path='/home/sagemaker-user/SageMaker/output/01_download_images_output.ipynb',
        dag=dag,
    )

    notebook_2 = ExecuteNotebookOperator(
        task_id='execute_notebook_2',
        notebook_path='/home/sagemaker-user/SageMaker/02_process_tfrecords.ipynb',
        output_path='/home/sagemaker-user/SageMaker/output/02_process_tfrecords_output.ipynb',
        dag=dag,
    )

    # Add other notebooks here in the same way

    end = DummyOperator(
        task_id='end',
        dag=dag,
    )

    start >> notebook_1 >> notebook_2 >> end
    """

    with open(f"{airflow_home}/dags/sagemaker_notebooks_dag.py", "w") as f:
        f.write(dag_content)

    print("Airflow setup completed successfully! Please restart the Jupyter server manually to apply the new configurations.")

# Instalar Airflow se necessário e configurar
install_airflow()
setup_airflow()


Airflow já está instalado.
DB: sqlite:////home/sagemaker-user/reproducible/airflow/airflow.db
[2024-06-04 01:41:01,937] {db.py:919} INFO - Creating tables
Initialization done

Arquivo de banco de dados criado com sucesso.
[2024-06-04 01:41:12,126] {manager.py:512} WARNING - Refused to delete permission view, assoc with role exists DAG Runs.can_create Admin
Recognized Database Authentications.
Error! User already exists admin


Restart the Jupyter server manually to apply the new configurations.
Airflow setup completed successfully! Please restart the Jupyter server manually to apply the new configurations.


In [2]:
with open("/home/sagemaker-user/reproducible/airflow/airflow.cfg", "w") as f:
    f.write("""
[core]
dags_folder = /home/ec2-user/SageMaker/airflow/dags
sql_alchemy_conn = sqlite:////home/ec2-user/SageMaker/airflow/airflow.db
executor = SequentialExecutor

[webserver]
web_server_port = 8080
""")

## Configurar Docker-Compose

In [3]:
!mkdir -p $HOME/bin
!curl -L "https://github.com/docker/compose/releases/download/1.29.2/docker-compose-$(uname -s)-$(uname -m)" -o $HOME/bin/docker-compose
!chmod +x $HOME/bin/docker-compose

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 12.1M  100 12.1M    0     0  88.0M      0 --:--:-- --:--:-- --:--:-- 88.0M


In [4]:
!export PATH=$HOME/bin:$PATH
!echo 'export PATH=$HOME/bin:$PATH' >> ~/.bashrc

In [5]:
!source ~/.bashrc

/bin/sh: 1: source: not found


In [6]:
!docker-compose --version

/bin/sh: 1: docker-compose: not found
